# 02 — Train

Runs the real fine-tuning pipeline (`src/train.py`) for one fold, for inspection. The full run behind `reports/experiments_report.md` was launched via:

```bash
python3 -m src.train --config configs/default.yaml
```

which trains `microsoft/deberta-v3-small` as a regressor across K stratified folds (config default: 3 folds / 2 epochs, see `configs/default.yaml` for why).

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from src.train import train_one_fold, load_config, seed_everything

cfg = load_config('../configs/default.yaml')
seed_everything(cfg['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
df = pd.read_csv('../data/train.csv')
skf = StratifiedKFold(n_splits=cfg['k_folds'], shuffle=True, random_state=cfg['seed'])
tr_idx, va_idx = next(skf.split(df, df['score'].astype(int)))
df_tr = df.iloc[tr_idx].reset_index(drop=True)
df_va = df.iloc[va_idx].reset_index(drop=True)
print('train:', len(df_tr), 'val:', len(df_va))

In [ ]:
path, val_qwk = train_one_fold(
    cfg, fold=0, df_train=df_tr, df_valid=df_va,
    text_col='full_text', target_col='score', device=device,
)
print('Best val QWK:', val_qwk, '-> saved to', path)